# Lesson 8: Serving and Next Steps

## Overview

In Lesson 7 you trained a transformer and used it to generate text. That workflow — call the model inside a Python session — works for experiments, but production inference needs more: predictable latency, no cold-start compilation, and portable model artifacts that can run without the original source code.

This lesson walks through the JAX inference pipeline, from a JIT-compiled forward pass to ahead-of-time (AOT) compilation, JAX-native export with `jax.export`, and TensorFlow SavedModel export with `jax2tf` when TensorFlow serving infrastructure is required.

**What you'll do:**

* Rebuild the Lesson 7 transformer and load a saved checkpoint with Orbax
* Wrap the forward pass in `jax.jit` and measure inference latency (first call vs subsequent calls)
* Use AOT compilation (`lower()` → `compile()`) to eliminate first-call compilation overhead
* Inspect the compiled StableHLO IR
* Batch multiple prompts and measure forward-pass throughput in tokens/sec
* Export the model to a portable JAX artifact with `jax.export`
* Convert to a TensorFlow SavedModel with `jax2tf` for TF Serving or Gemini Enterprise Agent Platform
* Compare all four paths: JIT, AOT compile, `jax.export`, and `jax2tf`

## From training to serving

This lesson compares four practical ways to move a trained JAX model toward serving, depending on your deployment target:

| Path | Format | Serving target | When to use |
| --- | --- | --- | --- |
| `jax.jit` | Cached in-process executable | Python server (FastAPI, Flask) | Simplest low-latency JAX serving path |
| AOT compile | Precompiled in-process executable | Python server startup / warmup | Avoid first-request compilation latency |
| `jax.export` | Serialized JAX export with StableHLO + metadata | Compatible JAX runtime for the exported platform(s) | JAX-native portable artifact |
| `jax2tf` | TF SavedModel | TF Serving, Gemini Enterprise Agent Platform, TFX | TensorFlow ecosystem |

This lesson demonstrates all four, starting from the same trained checkpoint.

## Requirements

The fixed NGC image and pinned workshop requirements provide:

- `jax`, `jaxlib` — core JAX
- `flax` — NNX modules (same model definition as Lesson 7)
- `orbax-checkpoint` — load the trained checkpoint
- `tensorflow` — for the `jax2tf` export path

## Setup

Import JAX, rebuild the model architecture from Lesson 7, and load the trained checkpoint.

In [ ]:
import os

os.environ["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

import html
import pathlib
import time
import warnings

import numpy as np
from IPython.display import HTML, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*ml_dtypes.*")
warnings.filterwarnings("ignore", message=".*JAX_PLATFORMS.*")

import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from flax import nnx

devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == "gpu"]

print(f"JAX version:     {jax.__version__}")
print(f"Default backend: {jax.default_backend()}")
print(f"GPU devices:     {gpu_devices}")

assert len(gpu_devices) >= 1, (
    f"This lesson needs at least 1 GPU. Found {len(gpu_devices)}. "
    f"Available devices: {devices}"
)


def block_tree(tree):
    return jax.block_until_ready(tree)


def show_table(headers, rows, title=None, aligns=None):
    aligns = aligns or ["left"] * len(headers)
    parts = ["<div style='font-family: system-ui; max-width: 980px;'>"]
    if title:
        parts.append(f"<h4 style='margin: 0 0 8px 0;'>{html.escape(title)}</h4>")
    parts.append("<table style='border-collapse: collapse; width: 100%; font-size: 13px;'>")
    parts.append("<thead><tr>")
    for h, a in zip(headers, aligns):
        parts.append(
            f"<th style='text-align:{a}; border-bottom:1px solid #d0d7de; padding:6px;'>"
            f"{html.escape(str(h))}</th>"
        )
    parts.append("</tr></thead><tbody>")
    for row in rows:
        parts.append("<tr>")
        for cell, a in zip(row, aligns):
            parts.append(
                f"<td style='text-align:{a}; border-bottom:1px solid #eef1f4; padding:6px;'>"
                f"{html.escape(str(cell))}</td>"
            )
        parts.append("</tr>")
    parts.append("</tbody></table></div>")
    display(HTML("\n".join(parts)))

## Rebuild the model

This is the same `TinyTransformer` from Lesson 7. We define it here so the notebook is self-contained, then load the trained weights from the Orbax checkpoint.

In [ ]:
VOCAB_SIZE = 256
D_MODEL = 256
NUM_HEADS = 4
FFN_DIM = 1024
NUM_LAYERS = 4
MAX_SEQ_LEN = 256


def causal_sdpa(query, key, value, **_):
    return jax.nn.dot_product_attention(query, key, value, is_causal=True)


class TransformerBlock(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, ffn_dim: int, rngs: nnx.Rngs):
        self.ln1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.attn = nnx.MultiHeadAttention(
            num_heads=num_heads,
            in_features=d_model,
            decode=False,
            attention_fn=causal_sdpa,
            rngs=rngs,
        )
        self.ln2 = nnx.LayerNorm(d_model, rngs=rngs)
        self.fc_up = nnx.Linear(d_model, ffn_dim, rngs=rngs)
        self.fc_down = nnx.Linear(ffn_dim, d_model, rngs=rngs)

    def __call__(self, x):
        x = x + self.attn(self.ln1(x))
        h = jax.nn.gelu(self.fc_up(self.ln2(x)))
        x = x + self.fc_down(h)
        return x


class TinyTransformer(nnx.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 ffn_dim: int, num_layers: int, max_seq_len: int,
                 rngs: nnx.Rngs):
        self.token_embed = nnx.Embed(vocab_size, d_model, rngs=rngs)
        self.pos_embed = nnx.Embed(max_seq_len, d_model, rngs=rngs)
        self.blocks = nnx.List([
            TransformerBlock(d_model, num_heads, ffn_dim, rngs=rngs)
            for _ in range(num_layers)
        ])
        self.final_norm = nnx.LayerNorm(d_model, rngs=rngs)
        self.lm_head = nnx.Linear(d_model, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, tokens):
        _, T = tokens.shape
        x = self.token_embed(tokens) + self.pos_embed(jnp.arange(T))
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        return self.lm_head(x)


param_count = sum(x.size for x in jax.tree.leaves(nnx.state(TinyTransformer(
    VOCAB_SIZE, D_MODEL, NUM_HEADS, FFN_DIM, NUM_LAYERS, MAX_SEQ_LEN,
    rngs=nnx.Rngs(0),
), nnx.Param)))
print(f"TinyTransformer: {param_count:,} parameters")

## Load the trained checkpoint

Load the Orbax checkpoint saved in Lesson 7. If you haven't run Lesson 7 yet, run it first — the checkpoint directory must exist.

In [ ]:
ckpt_dir = pathlib.Path("/tmp/jax-course/l7-checkpoints")

model = TinyTransformer(
    VOCAB_SIZE, D_MODEL, NUM_HEADS, FFN_DIM, NUM_LAYERS, MAX_SEQ_LEN,
    rngs=nnx.Rngs(0),
)
model_params = nnx.state(model, nnx.Param)
abstract_params = jax.tree.map(
    lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype), model_params
)

checkpointer = ocp.StandardCheckpointer()
restored_params = checkpointer.restore(ckpt_dir / "trained", abstract_params)

# Move to a single GPU (checkpoint may have multi-GPU sharding from L7)
single_device = gpu_devices[0]
restored_params = jax.device_put(restored_params, single_device)
nnx.update(model, restored_params)

print(f"\u2705 Checkpoint loaded from {ckpt_dir / 'trained'}")
print(f"Parameters: {sum(x.size for x in jax.tree.leaves(restored_params)):,}")

## Path 1: JIT inference

The simplest serving path — wrap the forward pass in `jax.jit`. The first call triggers compilation (cold start); subsequent calls reuse the cached executable.

For serving, we bake the model weights into a closure: `nnx.split` separates the model into a static graph definition and dynamic state, then a JIT-compiled function captures both and only takes `tokens` as input. This makes the compiled function self-contained — ideal for export.

In [ ]:
graphdef, model_state = nnx.split(model)

@jax.jit
def predict_jit(tokens):
    m = nnx.merge(graphdef, model_state)
    return m(tokens)

dummy_input = jnp.zeros((1, MAX_SEQ_LEN), dtype=jnp.int32)

# First call — includes compilation
start = time.perf_counter()
logits = block_tree(predict_jit(dummy_input))
first_call_ms = (time.perf_counter() - start) * 1000

# Subsequent calls — cached executable
times = []
for _ in range(100):
    start = time.perf_counter()
    logits = block_tree(predict_jit(dummy_input))
    times.append((time.perf_counter() - start) * 1000)

avg_ms = np.mean(times)

show_table(
    ["", "Latency (ms)"],
    [
        ("First call (compile + execute)", f"{first_call_ms:,.1f}"),
        ("Subsequent calls (avg of 100)", f"{avg_ms:.2f}"),
        ("Speedup", f"{first_call_ms / avg_ms:.0f}\u00d7"),
    ],
    title="JIT inference latency",
    aligns=["left", "right"],
)

print(f"\nOutput shape: {logits.shape} (batch=1, seq={MAX_SEQ_LEN}, vocab={VOCAB_SIZE})")

## Path 2: Ahead-of-time (AOT) compilation

`jax.jit` compiles on first call — that cold start is fine during development but a problem for serving. AOT compilation separates these steps:

<div style="font-family: Arial, sans-serif; max-width: 980px; line-height: 1.35; display: grid; grid-template-columns: 1fr 28px 1fr 28px 1fr 28px 1fr; gap: 10px; align-items: stretch; margin-top: 12px;">
  <div style="grid-column: 1; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">1. Python</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Define the function</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">You write a normal Python/JAX function, such as a model forward pass.</div>
  </div>

  <div style="grid-column: 2; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&rarr;</div>

  <div style="grid-column: 3; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">2. lower()</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Lower the computation</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">JAX traces the function for specific input shapes and dtypes and converts it into compiler IR.</div>
  </div>

  <div style="grid-column: 4; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&rarr;</div>

  <div style="grid-column: 5; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">3. StableHLO IR</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Inspect the portable program</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">StableHLO describes the computation in hardware-independent ops like <code>dot</code>, <code>reshape</code>, and <code>reduce</code>.</div>
  </div>

  <div style="grid-column: 6; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&rarr;</div>

  <div style="grid-column: 7; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">4. compile()</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Create the executable</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">XLA optimizes the StableHLO and produces a device-specific executable for GPU, TPU, or CPU.</div>
  </div>
</div>
<br>

StableHLO intermediate representation (IR) is the compiler-level representation of a JAX computation after Python code has been lowered into a portable, hardware-independent program. It describes operations such as matrix multiplies, reshapes, reductions, and control flow in a form that XLA can compile for different backends, including GPUs, TPUs, and CPUs. 

You compile once (at startup or offline for a fixed input shape), then execute many times without paying first-call compilation overhead. You can also inspect the StableHLO IR to see exactly what operations the compiler produced.

In [ ]:
# Bake weights into a closure — the compiled function only takes tokens
@jax.jit
def predict_closed(tokens):
    m = nnx.merge(graphdef, model_state)
    return m(tokens)

# Stage 1: Lower — trace the function and produce StableHLO
abstract_tokens = jax.ShapeDtypeStruct((1, MAX_SEQ_LEN), jnp.int32)

lowered = predict_closed.lower(abstract_tokens)
print(f"Lowered to StableHLO ({len(lowered.as_text()):,} chars)")

# Stage 2: Compile — produce device-specific executable
compiled = lowered.compile()
print(f"Compiled for: {jax.default_backend()}")

# Execute — no compilation overhead
start = time.perf_counter()
logits_aot = block_tree(compiled(dummy_input))
aot_first_ms = (time.perf_counter() - start) * 1000

times_aot = []
for _ in range(100):
    start = time.perf_counter()
    logits_aot = block_tree(compiled(dummy_input))
    times_aot.append((time.perf_counter() - start) * 1000)

avg_aot_ms = np.mean(times_aot)
max_diff_jit_aot = float(jnp.max(jnp.abs(logits - logits_aot)))

show_table(
    ["", "Latency (ms)"],
    [
        ("AOT first execution (no compile)", f"{aot_first_ms:.2f}"),
        ("AOT subsequent (avg of 100)", f"{avg_aot_ms:.2f}"),
        ("JIT first call (from above)", f"{first_call_ms:,.1f}"),
        ("Max |JIT − AOT|", f"{max_diff_jit_aot:.2e}"),
    ],
    title="AOT vs JIT latency",
    aligns=["left", "right"],
)

### Inspecting the StableHLO IR

`lowered.as_text()` shows the StableHLO program that will run on the device. This is the same intermediate representation used by XLA across GPUs, TPUs, and CPUs. It's useful for debugging, performance analysis, and understanding what the compiler actually sees.

StableHLO text can be very large, and a single line may contain a long constant or attribute. To avoid Jupyter's IOPub data-rate limit, the next cell saves the full IR to disk and prints only a bounded preview.

In [ ]:
hlo_text = lowered.as_text()

hlo_path = pathlib.Path("/tmp/jax-course/l8-stablehlo.mlir")
hlo_path.parent.mkdir(parents=True, exist_ok=True)
hlo_path.write_text(hlo_text)

MAX_LINES = 20
MAX_CHARS_PER_LINE = 160

lines = hlo_text.splitlines()
preview_lines = []
for line in lines[:MAX_LINES]:
    if len(line) > MAX_CHARS_PER_LINE:
        preview_lines.append(line[:MAX_CHARS_PER_LINE] + " ... [line truncated]")
    else:
        preview_lines.append(line)

print(f"StableHLO program: {len(lines):,} lines, {len(hlo_text):,} chars")
print(f"Full StableHLO saved to: {hlo_path}")
print("=" * 60)
print("\n".join(preview_lines))
print(
    f"\n... ({max(len(lines) - MAX_LINES, 0):,} more lines; "
    "long lines are truncated in this preview)"
)

## Batched inference

GPUs are most efficient when processing many inputs at once. Let's measure how forward-pass throughput scales with batch size. The same model function works, but each new input shape needs its own lowered/compiled executable, so we compile once per batch size.

The tokens/sec here counts tokens processed by a full forward pass. It is not the same as autoregressive generation throughput, where the model produces one new token at a time.

In [ ]:
batch_sizes = [1, 4, 16, 64]
results = []

for bs in batch_sizes:
    tokens_batch = jnp.zeros((bs, MAX_SEQ_LEN), dtype=jnp.int32)

    # Compile for this batch size
    lowered_bs = predict_closed.lower(
        jax.ShapeDtypeStruct((bs, MAX_SEQ_LEN), jnp.int32),
    )
    compiled_bs = lowered_bs.compile()

    # Warmup
    block_tree(compiled_bs(tokens_batch))

    # Measure
    times_bs = []
    for _ in range(50):
        start = time.perf_counter()
        block_tree(compiled_bs(tokens_batch))
        times_bs.append((time.perf_counter() - start) * 1000)

    avg_bs = np.mean(times_bs)
    tokens_per_sec = (bs * MAX_SEQ_LEN) / (avg_bs / 1000)
    results.append((bs, f"{avg_bs:.2f}", f"{tokens_per_sec:,.0f}"))

show_table(
    ["Batch size", "Latency (ms)", "Tokens/sec"],
    results,
    title="Batched inference throughput",
    aligns=["right", "right", "right"],
)

## Path 3: `jax.export` — portable StableHLO-based artifact

`jax.export` exports a jitted JAX function into an `Exported` object containing StableHLO plus the metadata needed to call it from another JAX process. The serialized bytes can be:

- Saved to disk and loaded in another process
- Called without the original Python model source code
- Exported for the current platform by default, or for explicit platforms with the `platforms=[...]` argument

This is the JAX-native deployment path — no TensorFlow dependency required.

In [ ]:
from jax import export

# Export the closure-based function — weights baked in, only tokens as input
exported = export.export(predict_closed)(
    jax.ShapeDtypeStruct((1, MAX_SEQ_LEN), jnp.int32),
)

print(f"Exported function: {exported.fun_name}")
print(f"Input shapes:  {exported.in_avals}")
print(f"Output shapes: {exported.out_avals}")
print(f"Exported platforms: {exported.platforms}")

# Serialize to bytes
blob = exported.serialize()
export_path = pathlib.Path("/tmp/jax-course/exports")
export_path.mkdir(parents=True, exist_ok=True)

export_file = export_path / "tiny_transformer_jax_export.bin"
export_file.write_bytes(blob)
print()
print(f"Serialized to {export_file} ({len(blob):,} bytes, {len(blob) / 1024:.0f} KB)")

# Deserialize and call — JAX is still required, but the model class/source is not
rehydrated = export.deserialize(export_file.read_bytes())

test_input = jnp.zeros((1, MAX_SEQ_LEN), dtype=jnp.int32)
logits_exported = block_tree(rehydrated.call(test_input))
print()
print(f"✅ Deserialized call succeeded — output shape: {logits_exported.shape}")

# Verify outputs match
diff_export = float(jnp.max(jnp.abs(logits_aot - logits_exported)))
print(f"Max difference from AOT: {diff_export:.2e}")

## Path 4: `jax2tf` — TensorFlow SavedModel

If your serving infrastructure uses TensorFlow (TF Serving, Gemini Enterprise Agent Platform, TFX pipelines), you can convert the JAX function to a TF SavedModel. `jax2tf` still lives under `jax.experimental`, but it is the standard JAX-to-TensorFlow interop path.

Native serialization is the default in current JAX releases, so the conversion embeds lowered StableHLO in the TensorFlow graph without passing `native_serialization=True`.

One important platform detail: this notebook runs JAX on CUDA, but the TensorFlow runtime in the JAX container may execute the SavedModel on CPU. To avoid a CUDA-exported module being called by TensorFlow on CPU, the cell below exports the `jax2tf` module for `("cpu",)`. If you serve with TensorFlow on GPU, export for `("cuda",)` and use a TensorFlow runtime with GPU/XLA support.

In [ ]:
import shutil

import tensorflow as tf
from jax.experimental import jax2tf


# Capture model_state as a closure so the TF function only takes tokens
def predict_for_tf(tokens):
    m = nnx.merge(graphdef, model_state)
    return m(tokens)

TF_EXPORT_PLATFORMS = ("cpu",)
tf_predict = jax2tf.convert(
    predict_for_tf,
    native_serialization_platforms=TF_EXPORT_PLATFORMS,
)

# Wrap in a tf.Module for SavedModel export. Autograph is unnecessary here because
# jax2tf already produced a TensorFlow-callable function.
module = tf.Module()
module.predict = tf.function(
    tf_predict,
    input_signature=[tf.TensorSpec(shape=(1, MAX_SEQ_LEN), dtype=tf.int32)],
    autograph=False,
)

# TF Serving expects a versioned model directory: <model_base>/<version>/saved_model.pb
savedmodel_base_dir = pathlib.Path("/tmp/jax-course/exports") / "tiny_transformer_savedmodel"
savedmodel_dir = savedmodel_base_dir / "1"
if savedmodel_base_dir.exists():
    shutil.rmtree(savedmodel_base_dir)

tf.saved_model.save(module, str(savedmodel_dir))
print(f"✅ SavedModel saved to {savedmodel_dir}")
print(f"Exported for TensorFlow platform(s): {TF_EXPORT_PLATFORMS}")

# Verify the SavedModel path produces the same logits as the JAX path.
with tf.device("/CPU:0"):
    tf_logits = module.predict(tf.zeros((1, MAX_SEQ_LEN), dtype=tf.int32))
diff_tf = np.max(np.abs(np.asarray(tf_logits) - np.asarray(logits_aot)))
print(f"Max difference from AOT: {diff_tf:.2e}")

# List saved files (in production, TF Serving loads this in a separate process)
for dirpath, _, filenames in os.walk(savedmodel_base_dir):
    for f in filenames:
        full = os.path.join(dirpath, f)
        size = os.path.getsize(full)
        print(f"  {os.path.relpath(full, savedmodel_base_dir):44s} {size:>10,} bytes")
print()
print(
    f"To serve: docker run -p 8501:8501 "
    f"--mount type=bind,source={savedmodel_base_dir},target=/models/transformer "
    "-e MODEL_NAME=transformer tensorflow/serving "
    "--xla_cpu_compilation_enabled=true"
)

## Comparison: all serving paths

Each path starts from the same trained checkpoint and should produce the same predictions. The differences are in portability, dependencies, deployment target, and whether the artifact can be used outside the original Python process.

In [ ]:
show_table(
    ["Path", "Format", "Dependencies", "Serving target", "Portable"],
    [
        ("jax.jit", "Cached in-process executable", "JAX", "Python server", "No"),
        ("AOT compile", "Compiled in-process executable", "JAX", "Python server startup / warmup", "No"),
        ("jax.export", "Serialized JAX export", "JAX runtime", "Compatible runtime for exported platform(s)", "Yes"),
        ("jax2tf", "TF SavedModel", "TensorFlow", "TF Serving / Gemini Enterprise Agent Platform", "Yes"),
    ],
    title="Serving path comparison",
)

# Show file sizes
sizes = []
export_size = os.path.getsize(export_file)
sizes.append(("jax.export", f"{export_size:,} bytes", f"{export_size / 1024:.0f} KB"))

sm_size = sum(
    os.path.getsize(os.path.join(dirpath, f))
    for dirpath, _, filenames in os.walk(savedmodel_base_dir)
    for f in filenames
)
sizes.append(("jax2tf SavedModel", f"{sm_size:,} bytes", f"{sm_size / 1024:.0f} KB"))

show_table(
    ["Export", "Size (bytes)", "Size (KB)"],
    sizes,
    title="Export file sizes",
    aligns=["left", "right", "right"],
)

## Summary

This lesson covered four practical paths from a trained JAX model to a serving-ready inference workflow:

* **`jax.jit`** is the simplest path — wrap and call. The first call compiles; subsequent calls are fast. Good for Python-based serving (FastAPI, Flask) where JAX is already installed.
* **AOT compilation** (`lower()` → `compile()`) separates compilation from execution. Compile once at startup, then serve without first-request compilation latency. `lowered.as_text()` shows the StableHLO IR for debugging and performance analysis.
* **`jax.export`** serializes a jitted function to a JAX-native artifact containing StableHLO and calling metadata. The resulting file can be loaded and called by a compatible JAX runtime without the original model code. By default it exports for the current platform; use `platforms=[...]` when you need an explicit target platform or multi-platform export.
* **`jax2tf`** converts the function to a TensorFlow SavedModel. Use this when your serving infrastructure is TensorFlow-based (TF Serving, Gemini Enterprise Agent Platform, TFX pipelines). Native serialization is the default in current JAX releases; choose `native_serialization_platforms` to match where TensorFlow will execute the model.

All four paths are meant to produce identical predictions from the same trained checkpoint. Choose based on your deployment constraints, not model architecture.

Official references:

* [JAX AOT compilation](https://docs.jax.dev/en/latest/aot.html)
* [jax.export guide](https://docs.jax.dev/en/latest/export/export.html)
* [JAX changelog notes for jax2tf native serialization](https://docs.jax.dev/en/latest/changelog.html)
* [Flax NNX basics](https://flax.readthedocs.io/en/latest/nnx/nnx_basics.html)
* [Orbax checkpoint guide](https://orbax.readthedocs.io/en/latest/guides/checkpoint/orbax_checkpoint_101.html)

## Course recap

Over eight lessons, you have:

1. **L1–L3**: Set up JAX, learned `jit` compilation, and profiled GPU execution
2. **L4**: Built a training loop from scratch — optimizer, loss, gradient updates
3. **L5**: Explored attention mechanisms — naive, SDPA, cuDNN fused attention
4. **L6**: Scaled to multiple GPUs with data parallelism — Mesh, NamedSharding, data-parallel sharding
5. **L7**: Combined everything into a transformer language model — Flax NNX, Orbax, generation
6. **L8**: Prepared the trained model for production — JIT, AOT, `jax.export`, `jax2tf`

## Next steps

You now have the core workflow for building with JAX on GPUs: array programming, jit, profiling, training loops, attention kernels, multi-GPU sharding, checkpointing, and export. The best next step is to turn these pieces into larger, messier projects: add KV-cache decoding and batched generation, train on a real tokenizer and dataset, experiment with mixed precision and quantization, scale beyond data parallelism into model/tensor parallelism, and build a small serving stack with monitoring and load tests. From here, the focus shifts from learning individual JAX features to making engineering choices: how to keep models fast, reproducible, memory-efficient, debuggable, and deployable.